In [ ]:
import json
import os
import re

folder_path = '/Users/pengyuan/Downloads/download'

with open('GENE-DESCRIPTION-JSON_WB.json', 'r') as f:
    gene_data = json.load(f)

gene_list = [item['gene_name'].lower() for item in gene_data['data']]
print(len(gene_list))

models = ['o1']

pdf_files = os.listdir(folder_path)
count = 0

for pdf_file in pdf_files:
    if pdf_file.endswith('.tei'):
        
        pdf_path = os.path.join(folder_path, pdf_file)
        # print(pdf_path)
        print(pdf_path)      
        pdf_text = extract_titleabstract_from_tei(pdf_path)
        pdf_text = pdf_text.lower()
        
        targeted_genes = []
        for gene_name in gene_list:
            if gene_name in pdf_text:
                targeted_genes.append(gene_name)
        
        print(targeted_genes)
        if len(targeted_genes)>0:
            pdf_text = extract_text_from_tei(pdf_path)
            task_name = pdf_file[:-4]+'_structured_gpt_responses_genes_targeted_UpdatedPrompt_o1.json'
            count += 1
            if not os.path.isfile(os.path.join(folder_path, task_name)):
                if len(pdf_text.split()) > 128000:
                    pdf_text = ' '.join(pdf_text.split()[:80000])
                print(len(pdf_text.split()))
                instruction = create_prompt_from_text(pdf_text, schema_json, ','.join(targeted_genes))
                gpt_response = information_extraction(instruction, models)
                responses = {}
                for model in models:
                    try:
                        responses[model] = json.loads(gpt_response[model])
                    except:
                        json_object_strings = re.findall(r'{\s*"gene_name":.*?}\s*}', gpt_response[model], re.DOTALL)
                        # Parse each JSON object
                        responses[model] = [json.loads(obj) for obj in json_object_strings]
                task_name = pdf_file[:-4]+'_structured_gpt_responses_genes_targeted_UpdatedPrompt_o1.json'
                with open(os.path.join(folder_path, task_name), 'w') as f:
                    json.dump(responses, f, indent=4)
print(count)

In [ ]:
from openai import OpenAI
import os
from lxml import etree

token = ''
client = OpenAI(api_key=token)

import pymupdf
import os

def extract_text_from_pdf(pdf_path):
    document = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    return text

def extract_text_from_tei(tei_path):

    tree = etree.parse(tei_path)
    root = tree.getroot()
    
    # Define TEI namespace
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    
    # Extract text from teiHeader (title, authors, etc.)
    title_el = root.xpath("//tei:titleStmt/tei:title", namespaces=ns)
    if title_el:
        title = " ".join(title_el[0].xpath(".//text()", namespaces=ns)).strip()
    else:
        title = "No title"
    
    # Extract abstract text
    abstract_paras = root.xpath("//tei:abstract//tei:p", namespaces=ns)
    abstract = " ".join(p.text for p in abstract_paras if p.text)
    
    # Extract body text (excluding anything in <back>)
    body_divs = root.xpath("//tei:body//tei:div", namespaces=ns)
    body_sections = []
    for div in body_divs:
        header = div.find("tei:head", namespaces=ns)
        section_title = " ".join(header.xpath(".//text()", namespaces=ns)).strip() if header is not None else ""
        paragraphs = div.findall(".//tei:p", namespaces=ns)
        paragraph_texts = [" ".join(p.xpath(".//text()", namespaces=ns)).strip() for p in paragraphs]
        section_block = f"{section_title}\n" + "\n".join(paragraph_texts) if section_title else "\n".join(paragraph_texts)
        body_sections.append(section_block)

    body = "\n".join(body_sections)
    
    # Combine all into a single text block
    output_text = f"Title: {title}\n\nAbstract: {abstract}\n\nPaper Body:\n{body}"

    # Print preview of the result
    return output_text  # Preview the first 1500 characters

def extract_titleabstract_from_tei(tei_path):

    tree = etree.parse(tei_path)
    root = tree.getroot()
    
    # Define TEI namespace
    ns = {"tei": "http://www.tei-c.org/ns/1.0"}
    
    # Extract text from teiHeader (title, authors, etc.)
    title_el = root.xpath("//tei:titleStmt/tei:title", namespaces=ns)
    if title_el:
        title = " ".join(title_el[0].xpath(".//text()", namespaces=ns)).strip()
    else:
        title = "No title"
    
    # Extract abstract text
    abstract_paras = root.xpath("//tei:abstract//tei:p", namespaces=ns)
    abstract = " ".join(p.text for p in abstract_paras if p.text)
    
    # Combine all into a single text block
    output_text = f"Title: {title}\n\nAbstract: {abstract}"

    # Print preview of the result
    return output_text  # Preview the first 1500 characters
    
# Define the schema as JSON for validation
schema_json = {
    "name": "gene_information_extraction",
    "schema": {"gene_name": "<Gene name>",
               "species": "<Species name>",
                "properties": {
                "biological_processes": {
                "type": "array",
                "description": "List of biological processes or programs the gene is directly involved in. Use actual Gene Ontology Biological Process terms.",
                "instructions": """1. When the gene plays an integral role in the biological process, based on experimental evidence in the paper, use this template: [Gene] is involved in the [Biological Process].  
                                2. When the gene's involvement in a biological process is not based on experimental evidence in the paper, and is based on some prediction or author suggestion, use this template: [Gene] is predicted to be involved in [biological process].  
                                3. When the mechanism relating the gene's activity to the Biological Process is not known, use this template: [Gene] acts upstream of or within [biological process]. 
                                """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "molecular_functions": {
                "type": "array",
                "description": "List of molecular-level activities performed by the gene. Use actual Gene Ontology Molecular Function terms.",
                "instructions": """1. When the gene enables a molecular function based on experimental evidence in the paper, use this template: [Gene] enables the [Molecular Function]. 
                                2. When the gene's molecular function is not based on experimental evidence in the paper, and is based on some prediction or author suggestion, use this template: [Gene] is predicted to enable the [molecular function].  
                                3. If the gene contributes to a molecular function as a member of a macromolecular complex (based on direct experimental evidence), in which the molecular function cannot be ascribed to an individual subunit of that complex, use this template: [Gene] contributes to [molecular function]. 
                                4. If the gene contributes to a molecular function as a member of a macromolecular complex (based on some prediction or author suggestion), in which the molecular function cannot be ascribed to an individual subunit of that complex, use this template: [Gene] is predicted to contribute to [molecular function].
                """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "interactions": {
                "type": "array",
                "description": "List the genes that the gene physically interacts with.",
                "instructions": """When there is experimental evidence in the paper for direct physical association or interaction between the gene product and other gene products, typically proteins. This interaction can involve binding, complex formation, or other direct physical contact that affects the function or activity of the gene products. These interactions are often studied using techniques such as co-immunoprecipitation, fluorescent staining, or the yeast two-hybrid system.  
                                1. When there is experimental evidence in the paper, use this template: [Gene] interacts with [Gene/Protein].  
                                2. When there is no experimental evidence in the paper and the interaction of the gene with other genes/proteins is based on predictions or author suggestions, use this template: [Gene] is predicted to interact with [Gene/Protein].
                """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "pathways": {
                "type": "array",
                "description": "List the genetic pathways that the gene is part of. Use Gene Ontology terms if they exist.",
                "instructions": """If the gene is part of a series of interactions between genes/proteins and other molecules within a cell that lead to a specific outcome or biological function. These interactions can involve a sequence of events, such as enzymatic reactions, that modify a molecule or a series of regulatory events that turn genes on or off, lead to a certain product or a change in the cell. 
                            1. When there is experimental evidence in the paper, use the template: [Gene] is part of [biological pathway].  
                            2. When there is no experimental evidence in the paper but only predictions that a gene is part of a biological pathway, use the template: [Gene] predicted to be part of [biological pathway].
                            """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "modifiers": {
                "type": "array",
                "description": "Genetic or chemical modifiers of disease.",
                "instructions": """List genetic modifiers or chemical modifiers that either ameliorate or exacerbate specific disease phenotypes (e.g., the paralysis phenotype). Use the WormBase Phenotype Ontology terms if they exist. Use the template, [Modifier] ameliorates or exacerbates the [disease  phenotype].
                                """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "gene_expression": {
                "type": "array",
                "description": "List the tissue and sub-cellular expressions of the gene. For sub-cellular locations, use Gene Ontology Cellular Component terms. For tissue expression, use the WormBase Anatomy Ontology terms.",
                "instructions": """For gene expression in tissue:  
                                If there is experimental evidence in the paper that the gene is expressed in any specific tissues, use template: [Gene] is expressed in [tissue].  
                                If expression of the gene in a tissue is based on some prediction or author suggestion, use this template: [Gene] is predicted to be expressed in [tissue].  
                                For gene expression in subcellular location:  
                                If the gene has been detected in a specific sub-cellular location based on experimental evidence in the paper, use this template: [Gene] is located in [subcellular location].  
                                If this is based on some prediction or author suggestion, use this template: [Gene] predicted to be located in [subcellular location].  
                                """,
                "items": {
                     "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            },
            "relevance_to_disease": {
                "type": "array",
                "description": "List those disease/s where the gene is directly implicated in the pathogenesis of disease or disease-like phenotypes, use Disease Ontology (DO) terms.",
                "instructions": """
                When there is direct evidence in the paper that the gene is directly implicated in the pathogenesis of the disease, use the template, [Gene] is used to study [Disease].
                When there is only a prediction or an author suggestion about the gene-disease relationship in the paper, ignore this.
                """,
                "items": {
                    "statement": "<follow templated sentence in instructions>",
                    "evidence_sentences": ["<supporting sentence 1>", "..."]
                }
            }
        },
        "required": [
            "biological_processes",
            "molecular_functions",
            "interactions",
            "pathways",
            "modifiers",
            "gene_expression",
            "relevance_to_disease"
        ],
        "additionalProperties": False
    }

}

def create_prompt_from_text(text, schema_json, gene_list):
    
    instruction = f"""
    Extract structured information from the experimental results of the paper content provided below.  
    Focus only on *C. elegans* genes in this list: {gene_list}.
    For each gene, produce a JSON object with the following structure. If a field has no information, set it as an empty array `[]`. Include the **exact sentences from the publication** that directly support each statement under `evidence_sentences`.

    {schema_json['schema']}
    
    Paper Content:
    {text}

    Make sure all fields are included. If a field has no information, set it as an empty array ([]). For each value, include all the actual sentences from the publication that directly support the statement. Ignore genes not related to *C. elegans*.
    Do not include the "description" and "instructions" field in the output json file.
    """
    return instruction

def information_extraction(instruction, models):
    gpt_responses = {}
    for model_name in models:
        print(model_name)
        response = client.chat.completions.create(
            model= model_name,  # Or the specific model you have access to
            messages=[
                {"role": "system", "content": "You are an assistant that extracts structured information from scientific papers based on a strict schema."},
                {"role": "user", "content": f"{instruction}"}
            ],
            #temperature=0,  # Adjust the temperature for more/less creativity
        )
        response_content = response.choices[0].message.content.strip()
        if response_content.startswith("```json") and response_content.endswith("```"):
            json_content = response_content[7:-3].strip()
        else:
            json_content = response_content.strip()
        gpt_responses[model_name] = json_content
    
    return gpt_responses
